# Pandas from First Principles
## Notebook 5: Sorting & Ranking

---

> **Series overview:** By the end of this notebook you will be able to sort DataFrames by one or more columns, retrieve the largest/smallest rows efficiently, and assign meaningful ranks to your data — skills you will use every day in data analysis.

In [1]:
import pandas as pd

# ── Datasets used throughout this notebook ──────────────────────────────────
employees = pd.DataFrame({
    'name':             ['Alice', 'Bob',       'Carol', 'Dave', 'Eve',       'Frank', 'Grace'],
    'department':       ['Eng',   'Marketing', 'Eng',   'HR',   'Marketing', 'Eng',   'HR'],
    'salary':           [85000,   72000,        91000,   68000,  75000,       95000,   64000],
    'age':              [28,      34,           26,      41,     30,          35,      27],
    'experience_years': [4,       8,            2,       15,     6,           10,      3],
})

students = pd.DataFrame({
    'name':    ['Riya', 'Arjun', 'Priya', 'Karan', 'Meera', 'Dev', 'Sana'],
    'math':    [88,     74,      91,      62,      85,      78,    95],
    'science': [79,     82,      88,      70,      91,      65,    84],
    'english': [92,     68,      85,      75,      80,      88,    77],
})

print('employees shape:', employees.shape)
print('students shape: ', students.shape)
print()
display(employees)
print()
display(students)

employees shape: (7, 5)
students shape:  (7, 4)



,name,department,salary,age,experience_years
0,Alice,Eng,85000,28,4
1,Bob,Marketing,72000,34,8
2,Carol,Eng,91000,26,2
3,Dave,HR,68000,41,15
4,Eve,Marketing,75000,30,6
5,Frank,Eng,95000,35,10
6,Grace,HR,64000,27,3


,name,math,science,english
0,Riya,88,79,92
1,Arjun,74,82,68
2,Priya,91,88,85
3,Karan,62,70,75
4,Meera,85,91,80
5,Dev,78,65,88
6,Sana,95,84,77


---
## 1. Introduction — Why Sorting Matters

Sorting is one of the most basic yet powerful operations in data analysis:

| Reason | Example |
|--------|---------|
| **Leaderboards** | Rank students or employees by score/salary |
| **Quick inspection** | Find the top/bottom performers at a glance |
| **Prerequisite for other ops** | Some algorithms require sorted input |
| **Report generation** | Alphabetical or chronological ordering |

### What is a Stable Sort?

A **stable sort** preserves the *original relative order* of rows that have **equal** values in the sort key.

```
Before sort (by dept):   After stable sort:    After unstable sort:
  Alice – Eng            Alice – Eng           Carol – Eng   ← order changed!
  Carol – Eng    →       Carol – Eng    vs.    Alice – Eng
  Frank – Eng            Frank – Eng           Frank – Eng
```

> **Good news:** Pandas `sort_values()` uses a **stable** merge-sort algorithm by default, so ties are always broken by the original row order. This matters when you sort by multiple columns.

---
## 2. `sort_values()` — Sort by Column Values

### What is it?
`sort_values()` reorders the rows of a DataFrame (or a Series) based on the values in one or more columns.

### Why use it?
It is the go-to method whenever you want to rank or order rows by a meaningful numeric or alphabetical criterion.

### Syntax
```python
df.sort_values(
    by,                  # str or list[str] — column(s) to sort by
    ascending=True,      # bool or list[bool] — sort direction per column
    na_position='last',  # 'last' or 'first' — where NaN values go
    ignore_index=False,  # if True, resets index to 0, 1, 2, …
    inplace=False,       # if True, modifies the DataFrame in place
)
```

### Return value
A **new DataFrame** with rows reordered (unless `inplace=True`).

### Best practices
- Always **assign the result** back: `df = df.sort_values(...)` or use `inplace=True` sparingly.
- Prefer `ignore_index=True` when the original index numbers are not meaningful after sorting.

### Common mistakes
```python
# ❌ WRONG — result is thrown away
df.sort_values(by='salary')
print(df)  # still unsorted!

# ✅ CORRECT — capture the result
df_sorted = df.sort_values(by='salary')
print(df_sorted)
```

In [4]:
# ── Single-column sort (ascending — default) ─────────────────────────────────
# Sort employees from lowest to highest salary
sorted_asc = employees.sort_values(by='salary')

print('Employees sorted by salary (low → high):')
display(sorted_asc[['name', 'department', 'salary']])
print()
print('Notice: the original index labels are preserved (0-6 are scrambled).')
print('Original employees DataFrame is unchanged:')
display(employees[['name', 'salary']])

Employees sorted by salary (low → high):


,name,department,salary
6,Grace,HR,64000
3,Dave,HR,68000
1,Bob,Marketing,72000
4,Eve,Marketing,75000
0,Alice,Eng,85000
2,Carol,Eng,91000
5,Frank,Eng,95000



Notice: the original index labels are preserved (0-6 are scrambled).
Original employees DataFrame is unchanged:


,name,salary
0,Alice,85000
1,Bob,72000
2,Carol,91000
3,Dave,68000
4,Eve,75000
5,Frank,95000
6,Grace,64000


### 2a. Descending Sort

Pass `ascending=False` to reverse the order.  
Most real-world rankings (top earners, highest scores) use descending order.

In [5]:
# ── Descending sort + reset index ────────────────────────────────────────────
# ignore_index=True gives a clean 0-based index in the result
top_earners = employees.sort_values(by='salary', ascending=False, ignore_index=True)

print('Employees sorted by salary (high → low), index reset:')
display(top_earners[['name', 'department', 'salary']])

Employees sorted by salary (high → low), index reset:


,name,department,salary
0,Frank,Eng,95000
1,Carol,Eng,91000
2,Alice,Eng,85000
3,Eve,Marketing,75000
4,Bob,Marketing,72000
5,Dave,HR,68000
6,Grace,HR,64000


### 2b. Multi-Column Sort

Pass a **list** to `by` and a matching **list** to `ascending`.

```python
df.sort_values(by=['col1', 'col2'], ascending=[True, False])
```

Pandas sorts by `col1` first; rows with the same `col1` value are then sorted by `col2`.

In [7]:
# ── Multi-column sort ────────────────────────────────────────────────────────
# Primary key  : department (A → Z)
# Secondary key: salary (high → low) within each department
multi_sorted = employees.sort_values(
    by=['department', 'salary'],
    ascending=[True, False],
    ignore_index=True,
)

print('Sorted by department (A-Z) then salary (high → low):')
display(multi_sorted[['name', 'department', 'salary']])

Sorted by department (A-Z) then salary (high → low):


,name,department,salary
0,Frank,Eng,95000
1,Carol,Eng,91000
2,Alice,Eng,85000
3,Dave,HR,68000
4,Grace,HR,64000
5,Eve,Marketing,75000
6,Bob,Marketing,72000


### 2c. Handling NaN Values — `na_position`

| `na_position` | Behaviour |
|---------------|----------|
| `'last'` (default) | NaN rows appear at the **end** |
| `'first'`          | NaN rows appear at the **beginning** |

This is important when your column has missing values — you choose whether unknowns float to the top or sink to the bottom of your sorted table.

In [8]:
# ── NaN position during sort ─────────────────────────────────────────────────
import numpy as np

employees_with_nan = employees.copy()
employees_with_nan.loc[2, 'salary'] = np.nan  # Carol has no salary recorded
employees_with_nan.loc[5, 'salary'] = np.nan  # Frank has no salary recorded

print('na_position="last"  (default):')
display(employees_with_nan.sort_values(by='salary', na_position='last')[['name', 'salary']].to_string())
print()
print('na_position="first":')
display(employees_with_nan.sort_values(by='salary', na_position='first')[['name', 'salary']].to_string())

na_position="last"  (default):


'    name   salary\n6  Grace  64000.0\n3   Dave  68000.0\n1    Bob  72000.0\n4    Eve  75000.0\n0  Alice  85000.0\n2  Carol      NaN\n5  Frank      NaN'


na_position="first":


'    name   salary\n2  Carol      NaN\n5  Frank      NaN\n6  Grace  64000.0\n3   Dave  68000.0\n1    Bob  72000.0\n4    Eve  75000.0\n0  Alice  85000.0'

---
## 3. `sort_index()` — Sort by Row Index Labels

### What is it?
`sort_index()` reorders rows by their **index labels** (the row numbers on the left), rather than by column values.

### Why use it?
After filtering or slicing, the index becomes non-contiguous and out of order. `sort_index()` restores a logical order without resetting index values.

### Syntax
```python
df.sort_index(ascending=True)
```

### When to use
- After `df.loc[[3, 1, 5, 0], :]` — rows arrive in custom order
- After concatenation when original indices from two DataFrames clash
- When the index carries meaningful information (e.g., timestamps)

### Common mistake
Confusing `sort_index()` with `reset_index()`. `sort_index()` **reorders** rows but keeps their labels; `reset_index()` **replaces** the index with 0, 1, 2, …

In [9]:
# ── sort_index() example ─────────────────────────────────────────────────────
# Simulate a scrambled index — pick rows in a non-sequential order
scrambled = employees.loc[[4, 1, 6, 0, 3, 2, 5]].copy()

print('Scrambled index (after unusual selection):')
print(scrambled[['name', 'salary']])
print()

# Restore original index order
restored = scrambled.sort_index(ascending=True)
print('After sort_index(ascending=True):')
print(restored[['name', 'salary']])
print()

# Descending index order
desc_index = scrambled.sort_index(ascending=False)
print('After sort_index(ascending=False):')
print(desc_index[['name', 'salary']])

Scrambled index (after unusual selection):
    name  salary
4    Eve   75000
1    Bob   72000
6  Grace   64000
0  Alice   85000
3   Dave   68000
2  Carol   91000
5  Frank   95000

After sort_index(ascending=True):
    name  salary
0  Alice   85000
1    Bob   72000
2  Carol   91000
3   Dave   68000
4    Eve   75000
5  Frank   95000
6  Grace   64000

After sort_index(ascending=False):
    name  salary
6  Grace   64000
5  Frank   95000
4    Eve   75000
3   Dave   68000
2  Carol   91000
1    Bob   72000
0  Alice   85000


---
## 4. `nlargest()` and `nsmallest()`

### What is it?
These methods return the **n rows** with the largest (or smallest) values in one or more columns — without sorting the entire DataFrame.

### Why use it instead of `sort_values().head()`?
For large DataFrames, `nlargest(n, col)` is **significantly faster** because it uses a partial-sort (heap-based) algorithm — O(N log n) instead of O(N log N).

### Syntax
```python
df.nlargest(n, columns, keep='first')
df.nsmallest(n, columns, keep='first')

series.nlargest(n, keep='first')
series.nsmallest(n, keep='first')
```

### Parameters
| Parameter | Type | Description |
|-----------|------|-------------|
| `n` | int | Number of rows to return |
| `columns` | str or list | Column(s) to compare |
| `keep` | `'first'`, `'last'`, `'all'` | How to handle ties |

### `keep` parameter explained
| Value | Behaviour when there is a tie |
|-------|-------------------------------|
| `'first'` (default) | Keep the first occurrence |
| `'last'` | Keep the last occurrence |
| `'all'` | Keep **all** tied rows even if result is > n rows |

### Best practices
- Use `nlargest` / `nsmallest` instead of `sort_values().head()` when you only need the top/bottom N rows.
- Use `keep='all'` when fairness matters (e.g., exam results with tied scores).

In [12]:
# ── nlargest() on a DataFrame ────────────────────────────────────────────────
top3_salary = employees.nlargest(3, 'salary')

print('Top 3 earners (nlargest):')
display(top3_salary[['name', 'department', 'salary']])
print()

# ── nsmallest() on a DataFrame ───────────────────────────────────────────────
bottom2_salary = employees.nsmallest(2, 'salary')

print('Bottom 2 earners (nsmallest):')
display(bottom2_salary[['name', 'department', 'salary']])

Top 3 earners (nlargest):


,name,department,salary
5,Frank,Eng,95000
2,Carol,Eng,91000
0,Alice,Eng,85000



Bottom 2 earners (nsmallest):


,name,department,salary
6,Grace,HR,64000
3,Dave,HR,68000


In [17]:
# ── nlargest() / nsmallest() on a Series ────────────────────────────────────
math_series = students.set_index('name')['math']

print('Top 3 math scores (Series.nlargest):')
display(math_series.nlargest(3).to_frame())
print()

print('Bottom 2 math scores (Series.nsmallest):')
display(math_series.nsmallest(2).to_frame())

Top 3 math scores (Series.nlargest):


,math
name,
Sana,95
Priya,91
Riya,88



Bottom 2 math scores (Series.nsmallest):


,math
name,
Karan,62
Arjun,74


In [20]:
# ── keep='all' example ───────────────────────────────────────────────────────
# Introduce a tie: give Arjun the same math score as Meera (85)
students_tie = students.copy()
students_tie.loc[1, 'math'] = 85  # Arjun now ties with Meera

print("keep='first' (default) — only first occurrence of tied value:")
display(students_tie.nlargest(3, 'math', keep='first')[['name', 'math']])
print()

print("keep='all' — all tied rows included even if > n rows:")
display(students_tie.nlargest(3, 'math', keep='all')[['name', 'math']])

keep='first' (default) — only first occurrence of tied value:


,name,math
6,Sana,95
2,Priya,91
0,Riya,88



keep='all' — all tied rows included even if > n rows:


,name,math
6,Sana,95
2,Priya,91
0,Riya,88


---
## 5. `rank()` — Assign Ranks to Values

### What is it?
`rank()` assigns a **numeric rank** to each value in a Series (or each value in a DataFrame column). Rank 1 = smallest by default.

### Why is it different from sorting?

| Operation | What it returns |
|-----------|-----------------|
| `sort_values()` | Reorders rows |
| `rank()` | Adds a new column of rank numbers — original row order unchanged |

### Syntax
```python
series.rank(
    method='average',  # how to handle ties
    ascending=True,    # True → lowest value gets rank 1
    na_option='nan',   # 'nan', 'keep', 'top', or 'bottom'
)
```

### The `method` parameter — handling ties

Suppose two values are tied for 2nd place.

| Method | Ranks assigned to tied values | Notes |
|--------|-------------------------------|-------|
| `'average'` (default) | 2.5, 2.5 | Mean of the tied positions |
| `'min'` | 2, 2 | Both get the **lower** rank |
| `'max'` | 3, 3 | Both get the **higher** rank |
| `'first'` | 2, 3 | First occurrence gets lower rank |
| `'dense'` | 2, 2 | Like `'min'` but **no gaps** in rank sequence |

#### `'dense'` vs `'min'` illustrated

```
Values:     [100, 85, 85, 70]

'min'   →   [1,   2,  2,  4]   ← gap: 3 is skipped
'dense' →   [1,   2,  2,  3]   ← no gap: densely packed
```

> **Rule of thumb:** Use `'dense'` for leaderboards where you don't want to skip a rank number.

### Best practices
- Use `ascending=False` when rank 1 should mean **best/highest**.
- Add the result as a new column so you keep the original data intact.

### Common mistake
```python
# ❌ Sorting and ranking are often confused:
df.sort_values('score')   # reorders rows, does NOT add rank numbers
df['score'].rank()        # adds rank numbers, does NOT reorder rows
```

In [23]:
# ── Comparing all rank methods side-by-side ──────────────────────────────────
# Introduce a tie to see the difference clearly
scores = pd.Series([100, 85, 85, 70, 60], name='score')

rank_comparison = pd.DataFrame({'score': scores})
for method in ['average', 'min', 'max', 'first', 'dense']:
    rank_comparison[method] = scores.rank(method=method)

print('Rank method comparison (ascending=True, so 60 → rank 1 by default):')
print(rank_comparison.to_string(index=False))
print()
print('Notice: dense has no gaps (no skipped rank numbers after a tie).')

Rank method comparison (ascending=True, so 60 → rank 1 by default):
 score  average  min  max  first  dense
   100      5.0  5.0  5.0    5.0    4.0
    85      3.5  3.0  4.0    3.0    3.0
    85      3.5  3.0  4.0    4.0    3.0
    70      2.0  2.0  2.0    2.0    2.0
    60      1.0  1.0  1.0    1.0    1.0

Notice: dense has no gaps (no skipped rank numbers after a tie).


In [24]:
# ── Ranking in descending order (rank 1 = highest value) ─────────────────────
# ascending=False → highest salary gets rank 1
salary_rank = employees['salary'].rank(method='dense', ascending=False)

print('Salary rank (1 = highest paid, dense method):')
for name, sal, rnk in zip(employees['name'], employees['salary'], salary_rank):
    print(f'  {name:<8}  salary={sal:>6}  rank={int(rnk)}')

Salary rank (1 = highest paid, dense method):
  Alice     salary= 85000  rank=3
  Bob       salary= 72000  rank=5
  Carol     salary= 91000  rank=2
  Dave      salary= 68000  rank=6
  Eve       salary= 75000  rank=4
  Frank     salary= 95000  rank=1
  Grace     salary= 64000  rank=7


In [25]:
# ── Adding rank as a new column to the DataFrame ─────────────────────────────
employees_ranked = employees.copy()
employees_ranked['salary_rank'] = employees_ranked['salary'].rank(
    method='dense', ascending=False
).astype(int)

# Sort by rank to display clearly
print('Employees with salary_rank column (sorted by rank):')
display_cols = ['name', 'department', 'salary', 'salary_rank']
print(
    employees_ranked[display_cols]
    .sort_values('salary_rank')
    .to_string(index=False)
)

Employees with salary_rank column (sorted by rank):
 name department  salary  salary_rank
Frank        Eng   95000            1
Carol        Eng   91000            2
Alice        Eng   85000            3
  Eve  Marketing   75000            4
  Bob  Marketing   72000            5
 Dave         HR   68000            6
Grace         HR   64000            7


---
## 6. Practical Examples

Let's put everything together with four real-world scenarios.

In [29]:
# ── Practical 1: Top 5 Highest-Paid Employees ────────────────────────────────
top5_paid = employees.nlargest(5, 'salary', keep='all')[['name', 'department', 'salary']]

print('=== Top 5 Highest-Paid Employees ===')
display(top5_paid)

=== Top 5 Highest-Paid Employees ===


,name,department,salary
5,Frank,Eng,95000
2,Carol,Eng,91000
0,Alice,Eng,85000
4,Eve,Marketing,75000
1,Bob,Marketing,72000


In [34]:
# ── Practical 2: Student Leaderboard — Ranked by Math Score ─────────────────
leaderboard = students.copy()
leaderboard['math_rank'] = leaderboard['math'].rank(method='dense', ascending=False).astype(int)
leaderboard = leaderboard.sort_values(by='math_rank', ignore_index=True)

print('=== Student Math Leaderboard ===')
display(leaderboard[['math_rank', 'name', 'math']])

=== Student Math Leaderboard ===


,math_rank,name,math
0,1,Sana,95
1,2,Priya,91
2,3,Riya,88
3,4,Meera,85
4,5,Dev,78
5,6,Arjun,74
6,7,Karan,62


In [36]:
# ── Practical 3: Sales Data — Sort by Region then Revenue ────────────────────
sales = pd.DataFrame({
    'region':  ['North', 'South', 'North', 'East', 'South', 'East', 'West', 'West'],
    'product': ['Widget', 'Gadget', 'Gadget', 'Widget', 'Widget', 'Gadget', 'Widget', 'Gadget'],
    'revenue': [45000, 38000, 52000, 41000, 29000, 61000, 33000, 47000],
})

sales_sorted = sales.sort_values(
    by=['region', 'revenue'],
    ascending=[True, False],
    ignore_index=True,
)

print('=== Sales Sorted by Region (A-Z) then Revenue (High → Low) ===')
display(sales_sorted)

=== Sales Sorted by Region (A-Z) then Revenue (High → Low) ===


,region,product,revenue
0,East,Gadget,61000
1,East,Widget,41000
2,North,Gadget,52000
3,North,Widget,45000
4,South,Gadget,38000
5,South,Widget,29000
6,West,Gadget,47000
7,West,Widget,33000


In [37]:
# ── Practical 4: Bottom 3 Performing Products by Revenue ─────────────────────
bottom3_products = sales.nsmallest(3, 'revenue')[['product', 'region', 'revenue']]

print('=== Bottom 3 Performing Products ===')
display(bottom3_products)

=== Bottom 3 Performing Products ===


,product,region,revenue
4,Widget,South,29000
6,Widget,West,33000
1,Gadget,South,38000


---
## 7. Practice Questions

Try each question below before looking at the answer key.

---













**Q1 [Easy]** Sort the `employees` DataFrame by `salary` in **descending** order.  
Display the `name` and `salary` columns only.

In [43]:
print('Employees DF:')
display(employees)
print()

employees.sort_values(by='salary',
                      ascending=False,
                      ignore_index=True)[['name','salary']]

Employees DF:


,name,department,salary,age,experience_years
0,Alice,Eng,85000,28,4
1,Bob,Marketing,72000,34,8
2,Carol,Eng,91000,26,2
3,Dave,HR,68000,41,15
4,Eve,Marketing,75000,30,6
5,Frank,Eng,95000,35,10
6,Grace,HR,64000,27,3


,name,salary
0,Frank,95000
1,Carol,91000
2,Alice,85000
3,Eve,75000
4,Bob,72000
5,Dave,68000
6,Grace,64000


**Q2 [Easy]** Use `nlargest()` to find the **top 3 earners** from `employees`.  
Display `name`, `department`, and `salary`.

In [46]:
display(employees)

employees.nlargest(3,'salary')[['name','department','salary']]

,name,department,salary,age,experience_years
0,Alice,Eng,85000,28,4
1,Bob,Marketing,72000,34,8
2,Carol,Eng,91000,26,2
3,Dave,HR,68000,41,15
4,Eve,Marketing,75000,30,6
5,Frank,Eng,95000,35,10
6,Grace,HR,64000,27,3


,name,department,salary
5,Frank,Eng,95000
2,Carol,Eng,91000
0,Alice,Eng,85000


**Q3 [Easy]** Sort the `students` DataFrame by `math` score in ascending order and **reset the index** so it starts from 0.

In [49]:
display(students)

students.sort_values(by='math',ignore_index=True,ascending=True)

,name,math,science,english
0,Riya,88,79,92
1,Arjun,74,82,68
2,Priya,91,88,85
3,Karan,62,70,75
4,Meera,85,91,80
5,Dev,78,65,88
6,Sana,95,84,77


,name,math,science,english
0,Karan,62,70,75
1,Arjun,74,82,68
2,Dev,78,65,88
3,Meera,85,91,80
4,Riya,88,79,92
5,Priya,91,88,85
6,Sana,95,84,77


**Q4 [Easy]** Rank the students by their `math` score so that **rank 1 = highest** score.  
Use `method='dense'`. Print each student's name and rank.

In [53]:
students['math_rank'] = students['math'].rank(method='dense',ascending=False).astype(int)
display(students.sort_values(by='math_rank',ascending=True).set_index('math_rank'))

,name,math,science,english
math_rank,,,,
1,Sana,95,84,77
2,Priya,91,88,85
3,Riya,88,79,92
4,Meera,85,91,80
5,Dev,78,65,88
6,Arjun,74,82,68
7,Karan,62,70,75


**Q5 [Medium]** Sort `employees` by `department` (A → Z) and then by `salary` (high → low) within each department. Display `name`, `department`, and `salary`.

In [55]:
display(employees)

employees.sort_values(by=['department','salary'],
                      ascending=[True,False])[['name','department','salary']]

,name,department,salary,age,experience_years
0,Alice,Eng,85000,28,4
1,Bob,Marketing,72000,34,8
2,Carol,Eng,91000,26,2
3,Dave,HR,68000,41,15
4,Eve,Marketing,75000,30,6
5,Frank,Eng,95000,35,10
6,Grace,HR,64000,27,3


,name,department,salary
5,Frank,Eng,95000
2,Carol,Eng,91000
0,Alice,Eng,85000
3,Dave,HR,68000
6,Grace,HR,64000
4,Eve,Marketing,75000
1,Bob,Marketing,72000



**Q6 [Medium]** Add a column `salary_rank` to `employees` where **rank 1 = highest salary**.  
Use `method='min'` this time. Display `name`, `salary`, and `salary_rank`.

In [61]:
employees['salary_rank'] = employees['salary'].rank(ascending=False,
                                                    method='min').astype(int)

display(employees.sort_values(by='salary_rank').set_index('salary_rank')[['name','salary']])

,name,salary
salary_rank,,
1,Frank,95000
2,Carol,91000
3,Alice,85000
4,Eve,75000
5,Bob,72000
6,Dave,68000
7,Grace,64000



**Q7 [Medium]** Find the **bottom 2 students** by their **total score** across all three subjects (`math + science + english`).  
Add a `total` column, then use `nsmallest()`.

In [65]:
display(students)

students['total_score'] = students['math']+students['science']+students['english']

display(students.nsmallest(2,'total_score'))

,name,math,science,english,math_rank,total_score
0,Riya,88,79,92,3,259
1,Arjun,74,82,68,6,224
2,Priya,91,88,85,2,264
3,Karan,62,70,75,7,207
4,Meera,85,91,80,4,256
5,Dev,78,65,88,5,231
6,Sana,95,84,77,1,256


,name,math,science,english,math_rank,total_score
3,Karan,62,70,75,7,207
1,Arjun,74,82,68,6,224


---
## Answer Key

In [ ]:
# ── Answer Q1 ────────────────────────────────────────────────────────────────
# Sort employees by salary descending
ans1 = employees.sort_values(by='salary', ascending=False)
print('Q1 — Employees sorted by salary (high → low):')
print(ans1[['name', 'salary']].to_string(index=False))

In [ ]:
# ── Answer Q2 ────────────────────────────────────────────────────────────────
# Top 3 earners using nlargest()
ans2 = employees.nlargest(3, 'salary')
print('Q2 — Top 3 earners:')
print(ans2[['name', 'department', 'salary']].to_string(index=False))

In [ ]:
# ── Answer Q3 ────────────────────────────────────────────────────────────────
# Sort students by math ascending and reset index
ans3 = students.sort_values(by='math', ascending=True, ignore_index=True)
print('Q3 — Students sorted by math score (low → high), index reset:')
print(ans3[['name', 'math']].to_string())

In [ ]:
# ── Answer Q4 ────────────────────────────────────────────────────────────────
# Rank students by math score: rank 1 = highest
math_rank = students['math'].rank(method='dense', ascending=False).astype(int)
print('Q4 — Student math ranks (1 = highest):')
for name, rank in zip(students['name'], math_rank):
    print(f'  {name:<8}  rank={rank}')

In [ ]:
# ── Answer Q5 ────────────────────────────────────────────────────────────────
# Sort by department (A-Z) then salary (high → low)
ans5 = employees.sort_values(
    by=['department', 'salary'],
    ascending=[True, False],
    ignore_index=True,
)
print('Q5 — Sorted by department (A-Z), salary (high → low):')
print(ans5[['name', 'department', 'salary']].to_string(index=False))

In [ ]:
# ── Answer Q6 ────────────────────────────────────────────────────────────────
# Add salary_rank column using method='min'
ans6 = employees.copy()
ans6['salary_rank'] = ans6['salary'].rank(method='min', ascending=False).astype(int)
print('Q6 — Employees with salary_rank (method=min, 1=highest):')
print(ans6[['name', 'salary', 'salary_rank']].sort_values('salary_rank').to_string(index=False))

In [ ]:
# ── Answer Q7 ────────────────────────────────────────────────────────────────
# Bottom 2 students by total score across all subjects
ans7 = students.copy()
ans7['total'] = ans7['math'] + ans7['science'] + ans7['english']
bottom2_students = ans7.nsmallest(2, 'total')
print('Q7 — Bottom 2 students by total score:')
print(bottom2_students[['name', 'math', 'science', 'english', 'total']].to_string(index=False))

---
## Quick Revision Table

| Method | Purpose | Key Parameter | Returns |
|--------|---------|---------------|--------|
| `sort_values(by=...)` | Sort rows by column values | `ascending`, `na_position`, `ignore_index` | New DataFrame |
| `sort_index()` | Sort rows by index labels | `ascending` | New DataFrame |
| `nlargest(n, col)` | Top N rows by value | `keep` (`'first'`/`'last'`/`'all'`) | New DataFrame |
| `nsmallest(n, col)` | Bottom N rows by value | `keep` | New DataFrame |
| `series.rank()` | Numeric rank per value | `method`, `ascending`, `na_option` | Series of floats |

### `rank()` method cheat-sheet

| Method | Tie behaviour | Use when … |
|--------|--------------|------------|
| `'average'` | Mean of tied positions | Statistical analysis |
| `'min'` | Both get lower rank (gap after) | Conservative ranking |
| `'max'` | Both get higher rank (gap before) | Pessimistic ranking |
| `'first'` | First occurrence wins | Strict ordering, no ties |
| `'dense'` | Both get lower rank, **no gap** | Leaderboards, competitions |

### Golden rules
1. `sort_values()` does **not** modify the original — always assign the result.
2. Use `nlargest` / `nsmallest` over `sort_values().head()` for performance on large data.
3. Use `rank()` when you need the rank number **as a column**, not just a sorted view.
4. Prefer `method='dense'` for user-facing leaderboards to avoid confusing rank gaps.

---
## Interview Questions

**IQ1.** *What is the difference between `sort_values()` and `rank()`?*

> `sort_values()` **reorders the rows** of a DataFrame based on column values but does not add any new column. `rank()` assigns a **numeric rank to each value** and returns a Series (or DataFrame) of rank numbers without changing the row order. They answer different questions: `sort_values` answers "give me the data in order", while `rank` answers "what position does each value hold?"

---

**IQ2.** *When would you use `nlargest()` instead of `sort_values().head()`?*

> Both produce the same result, but `nlargest()` is **more efficient** for large DataFrames because it uses a partial sort (heap algorithm) with O(N log n) complexity versus the full O(N log N) sort. Also, `nlargest` is more concise and communicates intent clearly. Use `sort_values().head()` only when you also need the full sorted DataFrame for other purposes.

---

**IQ3.** *Explain the difference between `rank(method='min')` and `rank(method='dense')` with an example.*

> Both assign the **lower rank** to all tied values, but they differ in what comes next.
> 
> ```
> Values:   [100, 85, 85, 70]
> min   →   [1,   2,  2,  4]   ← rank 3 is SKIPPED
> dense →   [1,   2,  2,  3]   ← no gap, ranks are consecutive
> ```
> 
> With `'min'`, tied rows both get rank 2, and the next distinct value jumps to rank 4 (the position it would occupy if all ties were listed separately). With `'dense'`, the next distinct value gets rank 3, keeping the sequence gapless. Use `'dense'` for leaderboards to avoid confusing users with skipped numbers.

---
## What's Next?

### Notebook 6 — Applying Functions

Now that you can **sort and rank** your data, the next step is to **transform** it. In Notebook 6 we will cover:

| Topic | What you'll learn |
|-------|------------------|
| `apply()` | Apply any Python function row-wise or column-wise |
| `map()` | Element-wise transformation on a Series |
| `applymap()` / `map()` on DataFrame | Apply a function to every cell |
| Lambda functions | Write concise one-liner transformations |
| `assign()` | Add computed columns in a readable pipeline style |

> **Keep going!** Sorting is the foundation; function application is where the real data transformation magic begins.